# Token contributions by category

Aggregate the reference-score token attributions within the category-labelled subsets and plot the most influential terms.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Prepare token table

In [ ]:
import os
import re
import ast
import pandas as pd
import numpy as np
import spacy
from nltk.corpus import stopwords
import nltk

INPUT_CSV = str(paths.sentence_run(4) / 'general/shap_output_all_folds.csv')
OUTPUT_DIR = str(paths.sentence_run(4) / 'general/')
OUTPUT_NAME = 'prepared_tokens_all_categories'
SHAP_COLUMN = 'shap_Ref'
MAX_TOKEN_LENGTH = 20
ROUND_ZERO = 2
MIN_LEMMA_COUNT = 3
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
nlp = spacy.load('en_core_web_sm')


def parse_list_like(col: pd.Series) -> pd.Series:

    def parse_entry(x):
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        if isinstance(x, str):
            s = x.strip()
            if s.startswith('[') and s.endswith(']'):
                try:
                    return ast.literal_eval(s)
                except Exception:
                    return [s]
            return [item.strip() for item in s.split(',') if item.strip()]
        return []
    return col.apply(parse_entry)


def is_clean_word(token: str) -> bool:
    if not isinstance(token, str):
        return False
    token = token.strip()
    if not token or token in {'[CLS]', '[SEP]'}:
        return False
    return re.fullmatch('[A-Za-z]+', token) is not None


def lemmatise(token: str) -> str:
    if not token:
        return token
    doc = nlp(token)
    return doc[0].lemma_


def main():
    if not os.path.exists(INPUT_CSV):
        raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    df = pd.read_csv(INPUT_CSV)
    needed = {'token', SHAP_COLUMN, 'true_names', 'pred_names'}
    if not needed.issubset(df.columns):
        missing = needed - set(df.columns)
        raise ValueError(f'Missing required columns: {missing}. Present: {list(df.columns)}')
    keep_cols = ['token', SHAP_COLUMN, 'true_names', 'pred_names']
    for col in ['text_id', 'sent_id', 'fold']:
        if col in df.columns:
            keep_cols.append(col)
    df = df[keep_cols].copy()
    df = df.dropna(subset=['token', SHAP_COLUMN])
    df['token'] = df['token'].astype(str)
    df['shap_value'] = pd.to_numeric(df[SHAP_COLUMN], errors='coerce')
    df = df.dropna(subset=['shap_value'])
    df['true_names'] = parse_list_like(df['true_names'])
    df['pred_names'] = parse_list_like(df['pred_names'])
    if ROUND_ZERO is not None:
        df = df[df['shap_value'].round(ROUND_ZERO) != 0]
    if df.empty:
        print('No rows remain after zero-rounding filter.')
        return
    df['token_lower'] = df['token'].str.lower()
    df = df[(df['token'].str.len() <= MAX_TOKEN_LENGTH) & df['token_lower'].apply(is_clean_word) & ~df['token_lower'].isin(stop_words)].copy()
    if df.empty:
        print('No tokens remain after basic cleaning.')
        return
    df['lemma'] = df['token_lower'].apply(lemmatise)
    counts = df['lemma'].value_counts()
    valid_lemmas = set(counts[counts >= MIN_LEMMA_COUNT].index)
    df = df[df['lemma'].isin(valid_lemmas)].copy()
    if df.empty:
        print('No lemmas meet the minimum count threshold.')
        return
    parquet_path = os.path.join(OUTPUT_DIR, f'{OUTPUT_NAME}.parquet')
    csv_path = os.path.join(OUTPUT_DIR, f'{OUTPUT_NAME}.csv')
    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)
    print(f' Prepared dataframe saved:\n- {parquet_path}\n- {csv_path}')
    print(f'Rows: {len(df):,} | Columns: {list(df.columns)}')
if __name__ == '__main__':
    main()


## Category token plots

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_PREPARED = str(paths.sentence_run(4) / 'general/prepared_tokens_all_categories.parquet')
OUTPUT_DIR = str(paths.FIGURES_DIR)
FILTER_MODE = 'has_category'
AGG_KIND = 'sum'
TOP_N = 10


def mask_has_category(series_of_lists: pd.Series, category: str) -> np.ndarray:
    return series_of_lists.apply(lambda lst: category in lst).to_numpy()


def apply_mode_mask(df: pd.DataFrame, category: str, mode: str) -> np.ndarray:
    if mode == 'has_category':
        return mask_has_category(df['true_names'], category)
    elif mode == 'tp_category':
        return mask_has_category(df['true_names'], category) & mask_has_category(df['pred_names'], category)
    elif mode == 'pred_category':
        return mask_has_category(df['pred_names'], category)
    else:
        raise ValueError(f'Unknown FILTER_MODE: {mode}')


def aggregate_by_lemma(df_cat: pd.DataFrame, kind: str='sum') -> pd.DataFrame:
    if kind == 'sum':
        out = df_cat.groupby('lemma', as_index=False)['shap_value'].sum().rename(
            columns={'shap_value': 'agg'},
        )
    elif kind == 'mean':
        out = df_cat.groupby('lemma', as_index=False)['shap_value'].mean().rename(
            columns={'shap_value': 'agg'},
        )
    else:
        raise ValueError("AGG_KIND must be 'sum' or 'mean'")
    return out


def plot_top_positive_tokens(df_agg: pd.DataFrame, output_path: str, n=10):
    top = df_agg[df_agg['agg'] > 0].nlargest(n, 'agg')
    if top.empty:
        print(' No positive SHAP tokens found.')
        return
    y = range(len(top))
    fig, ax = plt.subplots(figsize=(6, 0.5 * len(top)))
    sns.set_style('whitegrid')
    ax.barh(y, top['agg'], color='red', alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(top['lemma'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Aggregated SHAP', fontsize=10)
    for i, val in enumerate(top['agg']):
        ax.text(val + 0.01 * abs(val), i, f'{val:.2f}', va='center', ha='left', fontsize=7)
    ax.set_title('')
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f' Saved: {output_path}')


def main():
    if not os.path.exists(INPUT_PREPARED):
        raise FileNotFoundError(f'Prepared file not found: {INPUT_PREPARED}')
    df = pd.read_parquet(INPUT_PREPARED)
    if not {'lemma', 'shap_value', 'true_names', 'pred_names'}.issubset(df.columns):
        raise ValueError('Missing required columns.')
    all_categories = sorted(set((cat for sublist in df['true_names'].dropna() for cat in sublist)))
    print(f'Found {len(all_categories)} categories.')
    for category in all_categories:
        print(f'\n→ Processing: {category}')
        mask = apply_mode_mask(df, category, FILTER_MODE)
        df_cat = df[mask].copy()
        if df_cat.empty:
            print(f' No data for: {category}')
            continue
        df_agg = aggregate_by_lemma(df_cat, AGG_KIND)
        df_agg = df_agg[df_agg['agg'] > 0]
        if df_agg.empty:
            print(f' No positive SHAP tokens for: {category}')
            continue
        safe_cat = re.sub('\\W+', '_', category.lower()).strip('_')
        out_path = os.path.join(OUTPUT_DIR, f'shap_top_{safe_cat}_{FILTER_MODE}_{AGG_KIND}.png')
        plot_top_positive_tokens(df_agg, output_path=out_path, n=TOP_N)
if __name__ == '__main__':
    main()
